## libraries 

In [1]:
from pandasql import sqldf
pysqldf = lambda q: sqldf(q, globals())
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

## import Data

In [2]:
os.chdir(r'G:\$UBH@J!T\Data Analytics\New Data')
ecom= pd.read_csv('ecommerce_sales_data (2).csv')
pd.set_option('display.max_columns',None)
ecom.head()

,Order Date,Product Name,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53
2,2022-05-11,Tablet,Electronics,South,5,5865,502.73
3,2024-03-16,Mouse,Accessories,South,2,786,202.87
4,2022-09-10,Mouse,Accessories,West,1,509,103.28


## Cleaning the data

Showing missing values

In [3]:
ecom.isnull().mean()

Order Date      0.0
Product Name    0.0
Category        0.0
Region          0.0
Quantity        0.0
Sales           0.0
Profit          0.0
dtype: float64

so we see here there is no missing values

Showing duplicates values

In [4]:
ecom.loc[ecom.duplicated()==True,]

,Order Date,Product Name,Category,Region,Quantity,Sales,Profit


I have't seen any duplicates value in this data

In [5]:
ecom.dtypes

Order Date       object
Product Name     object
Category         object
Region           object
Quantity          int64
Sales             int64
Profit          float64
dtype: object

here we can see order date format is object so we should change this format into date and time format

In [6]:
ecom['Order Date']= pd.to_datetime(ecom['Order Date'])

In [7]:
ecom.dtypes

Order Date      datetime64[ns]
Product Name            object
Category                object
Region                  object
Quantity                 int64
Sales                    int64
Profit                 float64
dtype: object

## Export the clean data

In [11]:
ecom.to_csv('ecom_clean_data.csv')

## EDA by using SQL

## Sales performance

What is the total sales?

In [8]:
ecom.head(2)

,Order Date,Product Name,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53


In [9]:
pysqldf('''select sum("Sales") as total_sales
from ecom
''')

,total_sales
0,10667881


here we can see total product sales value

Which product generates the highest sales?

In [10]:
pysqldf('''select "Product Name" ,sum("Sales") as total_sales
from ecom
group by "Product Name"
order by "total_sales" desc
limit 2
''')

,Product Name,total_sales
0,Camera,1177381
1,Monitor,1160048


So here we can see 1st and 2nd higest sales product

Which category has the highest sales?

In [11]:
pysqldf('''
with "category_highest_sales" as (
select "Category" ,sum("Sales") as "total_sales"
from ecom
group by "Category"
)
select*
from "category_highest_sales" 
order by "total_sales" desc
limit 2
''')

,Category,total_sales
0,Electronics,5326074
1,Accessories,4247591


we can see here Electronics catagory is the higest sale from all catagories

Which region generates the most sales?

In [12]:
pysqldf('''
with "region_most_sales" as (
select "Region" ,sum("Sales") as "total_sales"
from ecom
group by "Region"
)
select*
from "region_most_sales"
order by "total_sales" desc
limit 2
''')

,Region,total_sales
0,West,2844450
1,East,2675110


so we see West is the better with total sales from East

In [ ]:
How do sales change over time (monthly/yearly)?

In [12]:
ecom.head(2)

,Order Date,Product Name,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53


How do sales change over time (monthly/yearly)?

In [13]:
pysqldf('''with monthly_sales as (
select 
strftime('%Y-%m', "Order Date") AS month,
        SUM(Sales) AS total_sales
        from ecom
        group by "month"
)
select* 
from monthly_sales
order by "month"
''')

,month,total_sales
0,2022-01,341544
1,2022-02,208775
2,2022-03,294660
3,2022-04,230624
4,2022-05,314295
5,2022-06,273851
6,2022-07,214627
7,2022-08,296242
8,2022-09,240211
9,2022-10,324989


## Profit Analysis

In [ ]:
Which product gives the highest profit?

In [14]:
ecom.head(2)

,Order Date,Product Name,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53


In [22]:
pysqldf('''select "Product Name", sum(Profit) as higest_profit
from ecom
group by "Product Name"
order by higest_profit desc
limit 3
''')

,Product Name,higest_profit
0,Camera,207630.99
1,Monitor,202028.17
2,Mouse,185763.69


Ws can see product camera gives the highest profit from the other products 

Which category is most profitable?

In [27]:
pysqldf('''with most_profitable_category as (
select Category , sum("Profit") as total_profit
from ecom
group by Category
)
select *
from most_profitable_category
order by total_profit desc
limit 2
''')

,Category,total_profit
0,Electronics,923185.59
1,Accessories,736084.74


here we can see Electronics gives the highest profit

Which region has the highest profit?

In [30]:
pysqldf('''with "highest_profit_region" as (
select Region , sum(Profit) as total_profit
from ecom 
group by Region 
) 
select*
from highest_profit_region
order by total_profit desc
limit 2
''')

,Region,total_profit
0,West,495358.73
1,East,464888.46


here we can see region wise west is the most highest profit from the others

Are there products with high sales but low profit?

In [32]:
ecom.head(2)

,Order Date,Product Name,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53


In [44]:
pysqldf('''select "Product Name" ,sum(Sales) as total_sales,
sum(Profit) as total_profit
from ecom
group by "Product Name"
having total_sales > 10000 and total_profit < 100
order by total_sales desc
''')

,Product Name,total_sales,total_profit


we see there is no data products with high sales overthen 10000 and low profit below from 100

## Product Performance

Which products sell the most quantity?

In [46]:
pysqldf('''select "Product Name", sum(Quantity) as total_quantity
from ecom
group by "Product Name"
order by total_quantity desc
limit 3
''')

,Product Name,total_quantity
0,Monitor,1876
1,Smartwatch,1807
2,Camera,1795


we can see most highest quantity sell of product is monitor from the others 

Which products generate losses?

In [48]:
pysqldf('''select "Product Name", sum(Profit) total_profit
from ecom
group by "Product Name"
having total_profit < 0
order by total_profit
limit 5
''')

,Product Name,total_profit


here we can see there is no product which generate negetive losses

What are the top 10 products by sales?

In [49]:
pysqldf('''select "Product Name" ,sum(Sales) as total_sales
from ecom
group by "Product Name"
order by total_sales desc
limit 10
''')

,Product Name,total_sales
0,Camera,1177381
1,Monitor,1160048
2,Printer,1094216
3,Mouse,1074398
4,Smartphone,1069681
5,Smartwatch,1049211
6,Keyboard,1024507
7,Tablet,1023928
8,Laptop,1005873
9,Headphones,988638


here we can see top 10 sales product

## Regional Performance

Helps businesses plan market strategy

In [ ]:
Which region has the highest sales?

In [51]:
ecom.head(2)

,Order Date,Product Name,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53


In [53]:
pysqldf(''' with highest_region_sales as(
select Region, sum(Sales) as total_sales
from ecom
group by Region
)
select*
from highest_region_sales 
order by total_sales desc
limit 2
''')

,Region,total_sales
0,West,2844450
1,East,2675110


here we see highest sales region is west

Which region has the lowest profit?

In [56]:
pysqldf('''with lowest_profit_Region as (
select Region, sum(Profit) as total_profit
from ecom
group by Region
)
select*
from lowest_profit_Region
order by total_profit 
limit 2
''')

,Region,total_profit
0,North,426314.75
1,South,458103.27


here we see lowest profit region is north

Which region sells the most products?

In [57]:
pysqldf('''with most_products_sales as (
select Region , sum(Quantity) as total_quantity
from ecom
group by Region
)
select*
from most_products_sales
order by total_quantity desc
limit 2
''')

,Region,total_quantity
0,West,4488
1,South,4361


we can see the west region sold the highest number of products

## Category Analysis

Important for inventory planning

Which category sells the most?

In [58]:
pysqldf('''with most_catagory_sales as (
select Category, sum(Sales) as total_sales
from ecom 
group by Category
)
select*
from most_catagory_sales 
order by total_sales desc
limit 3
''')

,Category,total_sales
0,Electronics,5326074
1,Accessories,4247591
2,Office,1094216


here we can see most sales catagory is Electronics

Which category has the highest profit margin?

In [75]:
ecom.head(2)

,Order Date,Product Name,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53


In [78]:
pysqldf('''with catagory_margen_profit as (
select Category ,sum(Profit) as total_profit,
sum(Sales) as total_sales,
sum(Profit)*1.0 / sum(Sales) as profit_margin
from ecom
group by Category
) 
select* 
from catagory_margen_profit
order by profit_margin desc
 ''')

,Category,total_profit,total_sales,profit_margin
0,Electronics,923185.59,5326074,0.173333
1,Accessories,736084.74,4247591,0.173295
2,Office,185394.88,1094216,0.169432


here we can see Electronics, Accessories both catagory are same profit_margin and we get higest profit margin from them  

In [ ]:
Which category performs poorly?

In [81]:
pysqldf(''' with poor_catagory as (
select Category ,sum(Profit) as total_profit
from ecom
group by Category
)
select*
from poor_catagory
order by total_profit 
''')

,Category,total_profit
0,Office,185394.88
1,Accessories,736084.74
2,Electronics,923185.59


The Office category shows the lowest profit among all categories

## This Analysis helps companies identify growth trends and support better business decisions making